# Water geometry optimization with CMA-ES and PySCF

This primary runnable example uses **PySCF** for B3LYP/6-31G* single-point energies and
**CMA-ES** for the geometry search. Both O-H bonds share one length;
the second variable is the H-O-H angle. Lengths are in Angstrom, angles in
degrees, and energies in Hartree. No energy-surrogate model is trained.

Install `requirements.txt`, select that environment's kernel, and run
all cells in order. See the [README](../README.md), [Chinese README](../README.zh-CN.md),
and [experiment background](../docs/experiment.md).

## 1. Imports and calculation settings

In [ ]:
import math
import warnings

import cma
import numpy as np
from pyscf import gto, dft, lib


In [ ]:
dft_keyword = "B3LYP"
basis_keyword = "6-31G*"
charge = 0
multiplicity = 1
cal_method = "R"  # R: restricted, U: unrestricted, RO: restricted open-shell.
max_memory = 4000  # MB
threads = 4
lib.num_threads(threads)
penalty = 999.999  # Hartree

init_coord = [1.0, 100.0]  # Shared O-H length (Angstrom), H-O-H angle (degrees).
angle_scale = 100.0
sigma0 = 0.25
seed = 2445
maxiter = 80


## 2. PySCF energy evaluation

The spin passed to PySCF is `multiplicity - 1`. A restricted calculation
is switched to unrestricted for non-singlet input. The SCF convergence
tolerance is `1e-8` Hartree and the maximum cycle count is 128.
The `(75, 302)` atom grid retains the original point counts, but does not
establish an integration scheme identical to Gaussian's.

Non-converged or non-finite energies and SCF numerical failures return a
high penalty with a warning. Invalid calculation settings raise errors so they
can be fixed, rather than being hidden as unsuccessful optimization candidates.

In [ ]:
def energy_calculate(atom_list, dft_keyword, basis_keyword, charge=0, multiplicity=1, cal_method="R"):
    if cal_method not in {"R", "U", "RO"}:
        raise ValueError("cal_method must be R, U, or RO.")
    method = "U" if cal_method == "R" and multiplicity != 1 else cal_method
    mol = gto.M(
        atom="\n".join(atom_list),
        charge=charge,
        spin=multiplicity - 1,
        basis=basis_keyword,
        unit="Angstrom",
        max_memory=max_memory,
        verbose=0,
    )
    scf = {"R": dft.RKS, "U": dft.UKS, "RO": dft.ROKS}[method](mol)
    scf.verbose = 0
    scf.max_cycle = 128
    scf.conv_tol = 1e-8
    scf.grids.atom_grid = (75, 302)
    scf.xc = dft_keyword
    try:
        value = float(scf.kernel())
    except (RuntimeError, np.linalg.LinAlgError) as exc:
        warnings.warn(f"Rejected PySCF calculation: {exc}", RuntimeWarning, stacklevel=2)
        return penalty
    if not scf.converged or not math.isfinite(value):
        warnings.warn("Rejected non-converged or non-finite PySCF energy.", RuntimeWarning, stacklevel=2)
        return penalty
    return value


In [ ]:
def energy(coord):
    bond = float(coord[0])
    angle = float(coord[1]) * angle_scale
    if not (math.isfinite(bond) and math.isfinite(angle) and bond > 0 and 0 < angle < 180):
        return penalty
    atom_list = ["O", f"H 1 {bond}", f"H 1 {bond} 2 {angle}"]
    return energy_calculate(atom_list, dft_keyword, basis_keyword, charge, multiplicity, cal_method)


## 3. Check the initial geometry

In [ ]:
initial_energy = energy([init_coord[0], init_coord[1] / angle_scale])
if initial_energy >= penalty:
    raise RuntimeError("The initial PySCF calculation failed; check the settings and warnings.")
print(f"Initial energy: {initial_energy:.12f} Hartree")


## 4. CMA-ES optimization

Search `[bond_length, angle / 100]`. The objective restores the angle to degrees.
Bounds are 0.5-2.0 Angstrom and 30-170 degrees, with a default budget of 80
iterations. This water example evaluates candidates serially.

In [ ]:
# Optimize the shared bond length and the scaled angle.
es = cma.CMAEvolutionStrategy(
    [init_coord[0], init_coord[1] / angle_scale],
    sigma0,
    {
        "seed": seed,
        "bounds": [[0.5, 30.0 / angle_scale], [2.0, 170.0 / angle_scale]],
        "maxiter": maxiter,
        "tolfun": 1e-8,
        "verb_log": 0,
        "verb_disp": 10,
    },
)
es.optimize(energy)


## 5. Result

In [ ]:
if es.result.xbest is None or not math.isfinite(es.result.fbest) or es.result.fbest >= penalty:
    raise RuntimeError("No successful energy evaluations; check the calculation settings and warnings.")

bond = float(es.result.xbest[0])
angle = float(es.result.xbest[1]) * angle_scale
print(f"Shared O-H bond: {bond:.8f} Angstrom")
print(f"H-O-H angle:    {angle:.8f} degrees")
print(f"Energy:         {es.result.fbest:.12f} Hartree")
print(f"Evaluations:    {es.result.evaluations}")
print(f"Stop reasons:   {dict(es.stop())}")


Inspect the stopping conditions; `maxiter` means the iteration budget was exhausted.
The [experiment notes](../docs/experiment.md) explain coordinate representation,
earlier surrogate-energy experiments, and their conclusions. The
[validation record](../docs/validation.md) includes a complete PySCF run.
The earlier Gaussian backend is kept in [legacy/gaussian](../legacy/gaussian/README.md).
This notebook contains no saved execution output; run it to obtain fresh results.